# Part 1: Ready Made vs Custom Made Data

# Part 2: Find Researchers using the OpenAlex API

In [ ]:
import pandas as pd
from dotenv import load_dotenv
import requests
import os
from datetime import datetime
from time import sleep
import backoff # https://pypi.org/project/backoff/
from tqdm.contrib.concurrent import thread_map
from itertools import chain
import pickle
import json

d:\6_semester\02467_CSS\02467_venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [25]:
# ==========================
#   Load the scraped names
# ==========================
with open("scraped_names.json", "r", encoding="utf-8") as f:
    scraped_names = json.load(f)

scraped_names = scraped_names[3:4]
len(scraped_names)

1

In [26]:
scraped_names

['Dietram A. Scheufele']

In [27]:
# ==========================
#    Method for logging 
# ==========================
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
LOG_PATH = f"./logs/week3_logs/log_{timestamp}.txt"

def log_line(identifier, logText):
    """
    identifier: something that identifies the logText preferably uniquely for debugging.
    """
    with open(LOG_PATH, "a", encoding="utf-8") as f:
        f.write(f"{identifier}\t{logText}\n")
        f.flush()


# ====================================
# -- API query method using backoff --
# ====================================
# Retry on all RequestExceptions, exponential backoff with jitter.
@backoff.on_exception(
    backoff.expo,
    requests.exceptions.RequestException,
    max_tries=7,                     # default infinite — better to cap
    max_time=60,                     # stop after 60 sec
)
def get_with_backoff(url, **kwargs):
    resp = requests.get(url, **kwargs)

    if resp.status_code in (429, 500, 502, 503, 504):
        log_line(resp.status_code, resp.text)
        resp.raise_for_status()

    return resp

In [28]:
load_dotenv()
API_KEY = os.getenv("OPENALEX_API_KEY")

AUTHOR_BASE = "https://api.openalex.org/authors"

def getAuthors(scraped_names):
    names = "|".join(scraped_names)

    params = {
            "filter": f"display_name.search:{names}",
            "select": "id, display_name, works_api_url, summary_stats, last_known_institutions",
            "per_page": 200,
            "api_key": API_KEY,
            "cursor":"*"
        }

    results = []
    max_pages = 50 #safety cap
    pages = 0

    # Loop to query all the pages of works by an author
    while True:

        if pages >= max_pages:
            print(f"Stopped after reaching safety cap of {max_pages} pages.")
            log_line(pages, f"Stopped after reaching safety cap of {max_pages} pages.")
            break

        r = get_with_backoff(AUTHOR_BASE, params=params, timeout=60).json()

        page_results = r.get("results")
        if not page_results:
            break  # done
        results.extend(page_results)

        next_cursor = r.get("meta", {}).get("next_cursor")

        if not next_cursor:
            break  # done

        params["cursor"] = next_cursor
        pages += 1
        sleep(0.2) # polite pause; adjust according to hit rate limits

    return results


# Parallel processing of API queries using tqdm
chunks = [scraped_names[i:i+25] for i in range(0, len(scraped_names), 25)]
all_results = thread_map(getAuthors, chunks, max_workers=8, chunksize=1, desc="Fetching")

Fetching: 100%|██████████| 1/1 [00:02<00:00,  2.92s/it]


In [29]:
all_results

[[{'id': 'https://openalex.org/A5064753440',
   'display_name': 'Dietram A. Scheufele',
   'works_api_url': 'https://api.openalex.org/works?filter=author.id:A5064753440',
   'summary_stats': {'2yr_mean_citedness': 8.466666666666667,
    'h_index': 76,
    'i10_index': 205},
   'last_known_institutions': [{'id': 'https://openalex.org/I1304256225',
     'ror': 'https://ror.org/03ydkyb10',
     'display_name': 'University of Wisconsin System',
     'country_code': 'US',
     'type': 'education',
     'lineage': ['https://openalex.org/I1304256225']},
    {'id': 'https://openalex.org/I135310074',
     'ror': 'https://ror.org/01y2jtd41',
     'display_name': 'University of Wisconsin–Madison',
     'country_code': 'US',
     'type': 'education',
     'lineage': ['https://openalex.org/I135310074']},
    {'id': 'https://openalex.org/I29680605',
     'ror': 'https://ror.org/05cb4rb43',
     'display_name': 'Morgridge Institute for Research',
     'country_code': 'US',
     'type': 'nonprofit',
 

In [23]:
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

flat_results = list(chain.from_iterable(all_results))
df = pd.DataFrame(flat_results)
df[df["id"] == "https://openalex.org/A5123686599"]["summary_stats"]

1697    {'2yr_mean_citedness': 0.0, 'h_index': 0, 'i10...
Name: summary_stats, dtype: object

# Part 3: Collect Research Articles